In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.storage import InMemoryStore
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

# 1. Load environment
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# 1. Prepare documents
docs = [
    Document(page_content="LangChain helps build LLM-powered apps with memory and agents.", metadata={"id": "1"}),
    Document(page_content="Agents in LangChain use tools to answer questions.", metadata={"id": "2"})
]

# 2. Setup child splitter
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

# 3. Setup vectorstore — note: don't pre-populate with the full parent docs;
# ParentDocumentRetriever handles child-chunk embedding internally via add_documents()
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(["placeholder"], embedding)  # see note below
vectorstore.delete([vectorstore.index_to_docstore_id[0]])   # remove placeholder

# 4. Parent retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=InMemoryStore(),  # Stores parent docs
    child_splitter=child_splitter
)

# 5. Add documents (this is what actually populates the vectorstore with child chunks)
retriever.add_documents(docs)

# 6. Retrieve — get_relevant_documents is deprecated, use invoke()
results = retriever.invoke("What are agents?")
for doc in results:
    print("📄 Retrieved Doc:", doc.page_content)

📄 Retrieved Doc: Agents in LangChain use tools to answer questions.
📄 Retrieved Doc: LangChain helps build LLM-powered apps with memory and agents.
